# Part 1.3 — Matched CSV versus Snappy-Parquet benchmark

**Objective:** support the Part 1 storage choice with reproducible measurements rather than a generic claim about file formats.

**Experimental control:** choose five complete calendar years (2008–2012) from the *same* verified Financials source, copy their Parquet files to Colab local disk, generate an equivalent CSV representation, verify row/value equivalence and measure size plus repeated warm reads. No downloaded FDIC data or benchmark CSV is committed to GitHub.

The results quantify the **selected subset**, not necessarily the complete 1984–2026 panel. Warm-read timings are platform-dependent; no claim is made about uncached I/O.

## Step 1 — Restore code and dependencies

In [1]:
# Run in Google Colab after uploading the Part 1 source tree to GitHub.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/NguyenDuong0508/11230945_BFT65B_Final_Research_BigData.git"
REPO_DIR = Path("/content/11230945_BFT65B_Final_Research_BigData")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import pandas, pyarrow, requests
print("Python:", sys.version.split()[0], "| pandas:", pandas.__version__,
      "| PyArrow:", pyarrow.__version__, "| requests:", requests.__version__)

Working directory: /content/11230945_BFT65B_Final_Research_BigData
Python: 3.13.18 | pandas: 2.2.3 | PyArrow: 20.0.0 | requests: 2.32.5


## Step 2 — Connect Drive and select the same full data snapshot

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DATA_BASE = Path("/content/drive/MyDrive/BFT65B_FDIC")
SMOKE_ROOT = DATA_BASE / "smoke_2023Q1"
FULL_ROOT = DATA_BASE / "full_1984Q1_2026Q2"
DATA_BASE.mkdir(parents=True, exist_ok=True)
print("Persistent dataset location:", DATA_BASE)
print("The data are stored outside the Git repository and must NOT be committed.")

Mounted at /content/drive
Persistent dataset location: /content/drive/MyDrive/BFT65B_FDIC
The data are stored outside the Git repository and must NOT be committed.


## Step 3 — Benchmark five complete years
The benchmark independently verifies the whole acquisition snapshot before comparing formats. Local copies avoid attributing Google Drive mount latency to Parquet. The scratch CSV and Parquet copies are removed automatically at the end.

In [3]:
subprocess.run([
    sys.executable, "-m", "part1.benchmark", "--data-root", str(FULL_ROOT),
    "--years", "2008", "2009", "2010", "2011", "2012",
    "--repeats", "5", "--workdir", "/content",
    "--json-out", str(FULL_ROOT / "evidence" / "part1_benchmark.json"),
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'part1.benchmark', '--data-root', '/content/drive/MyDrive/BFT65B_FDIC/full_1984Q1_2026Q2', '--years', '2008', '2009', '2010', '2011', '2012', '--repeats', '5', '--workdir', '/content', '--json-out', '/content/drive/MyDrive/BFT65B_FDIC/full_1984Q1_2026Q2/evidence/part1_benchmark.json'], returncode=0)

## Step 4 — Review measured outputs and interpret the experiment
Storage ratio and read performance are distinct: a smaller file is not automatically faster for every workload. Both full-width and four-column scans are recorded.

In [4]:
import json
import pandas as pd
outcome = json.loads((FULL_ROOT / "evidence" / "part1_benchmark.json").read_text(encoding="utf-8"))
print("Acquisition years used:", outcome["years"])
print("Rows / columns:", outcome["rows"], "/", outcome["columns"])
print("Data equivalence:", outcome["equivalence_check"])
print("Method:", outcome["method"])
display(pd.DataFrame([outcome["storage"]]).T.rename(columns={0: "Measured value"}))
read_rows = [{"workload": name, **values}
             for name, values in outcome["read_time_seconds"].items()]
display(pd.DataFrame(read_rows))

Acquisition years used: [2008, 2009, 2010, 2011, 2012]
Rows / columns: 157851 / 33
Data equivalence: PASS: all fields, all rows (numeric tolerance 1e-7)
Method: same FDIC data; matched columns and rows; one warm-up then median of 5 reads; local Colab disk; OS cache not disabled; end-to-end download time not measured; pandas/Arrow readers


,Measured value
csv_to_parquet_size_ratio,1.435400e+00
equivalent_csv_bytes,3.829553e+07
parquet_saving_pct,3.033000e+01
snappy_parquet_bytes,2.667884e+07


,workload,max_seconds,median_seconds,min_seconds
0,csv_all_fields,1.295107,1.253146,0.918880
1,csv_four_fields,0.488908,0.472238,0.454159
2,parquet_all_fields,0.214793,0.159099,0.158543
3,parquet_four_fields,0.033441,0.032416,0.032089


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### Handoff and limitations

- Save the executed notebook (with outputs) back to the Git repository **after confirming outputs contain no downloaded records**. The JSON metrics remain in Google Drive and can be cited in the later report.
- Partition pruning and lazy-query optimization are analyzed in **Part 3** using Polars and Spark. This Part 1 benchmark only establishes the physical storage trade-off.
- The baseline comparison is **CSV representation of identical FDIC data**, not an independently obtained original CSV export.
- Avoid declaring a speedup without interpreting the actual medians reported above.